In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train_df = pd.read_csv('/kaggle/input/diabetes-risk/train.csv')
val_df = pd.read_csv('/kaggle/input/diabetes-risk/val.csv')
test_df = pd.read_csv('/kaggle/input/diabetes-risk/test.csv')

In [ ]:
# prepare features and target

identifier_cols = ['County', 'State']
target_col = 'target'
feature_cols = [col for col in train_df.columns if col not in identifier_cols + [target_col]]

print("number of features", len(feature_cols))

train_identifiers = train_df[identifier_cols]
val_identifiers = val_df[identifier_cols]
test_identifiers = test_df[identifier_cols]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

full_df = pd.concat([train_df, val_df, test_df], axis=0).reset_index(drop=True)

In [ ]:
# feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# model training

models = {}
predictions = {}

# linear regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
models['Linear Regression'] = lr_model
predictions['Linear Regression'] = {
    'train': lr_model.predict(X_train_scaled),
    'val': lr_model.predict(X_val_scaled),
    'test': lr_model.predict(X_test_scaled)
}

# random forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
models['Random Forest'] = rf_model
predictions['Random Forest'] = {
    'train': rf_model.predict(X_train),
    'val': rf_model.predict(X_val),
    'test': rf_model.predict(X_test)
}

# SVM
svm_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svm_model.fit(X_train_scaled, y_train)
models['SVM'] = svm_model
predictions['SVM'] = {
    'train': svm_model.predict(X_train_scaled),
    'val': svm_model.predict(X_val_scaled),
    'test': svm_model.predict(X_test_scaled)
}

In [ ]:
# evaluate models

results = []
for model_name in ['Linear Regression', 'Random Forest', 'SVM']:
    for dataset_name, y_true in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
        y_pred = predictions[model_name][dataset_name.lower()]
        
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        
        results.append({
            'Model': model_name,
            'Dataset': dataset_name,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2
        })

results_df = pd.DataFrame(results)
print("model performance summary")
print(results_df.to_string(index=False))

# save results
results_df.to_csv('model_performance_results.csv', index=False)

In [ ]:
# feature importance analysis

# linear regression
print("linear regression - top 15 feature coefficients")
lr_coef = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(lr_coef.head(15).to_string(index=False))
lr_coef.to_csv('linear_regression_coefficients.csv', index=False)

# random forest
print("random forest - top 15 feature importances")
rf_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(rf_importance.head(15).to_string(index=False))
rf_importance.to_csv('random_forest_feature_importance.csv', index=False)

In [ ]:
# visualizations

# model performance comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['RMSE', 'MAE', 'R2']
for idx, metric in enumerate(metrics):
    pivot_data = results_df.pivot(index='Model', columns='Dataset', values=metric)
    pivot_data.plot(kind='bar', ax=axes[idx], rot=45)
    axes[idx].set_title(f'{metric} Comparison Across Models', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(metric)
    axes[idx].legend(title='Dataset')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# random forest feature importance
fig, ax = plt.subplots(figsize=(12, 8))
top_features = rf_importance.head(20)
ax.barh(range(len(top_features)), top_features['Importance'].values)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values)
ax.set_xlabel('Importance Score', fontweight='bold')
ax.set_title('Top 20 Features - Random Forest Importance', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('random_forest_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# linear regression coefficients
fig, ax = plt.subplots(figsize=(12, 8))
top_coef = lr_coef.head(20)
colors = ['red' if x < 0 else 'green' for x in top_coef['Coefficient']]
ax.barh(range(len(top_coef)), top_coef['Coefficient'].values, color=colors, alpha=0.7)
ax.set_yticks(range(len(top_coef)))
ax.set_yticklabels(top_coef['Feature'].values)
ax.set_xlabel('Coefficient Value', fontweight='bold')
ax.set_title('Top 20 Features - Linear Regression Coefficients', 
             fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('linear_regression_coefficients.png', dpi=300, bbox_inches='tight')
plt.show()

# actual vs. predicted risk
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, model_name in enumerate(['Linear Regression', 'Random Forest', 'SVM']):
    y_pred = predictions[model_name]['test']
    axes[idx].scatter(y_test, y_pred, alpha=0.5, s=30)
    axes[idx].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
                   'r--', lw=2, label='Perfect Prediction')
    axes[idx].set_xlabel('Actual Risk', fontweight='bold')
    axes[idx].set_ylabel('Predicted Risk', fontweight='bold')
    axes[idx].set_title(f'{model_name} (R² = {r2_score(y_test, y_pred):.3f})', 
                       fontsize=12, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('actual_vs_predicted_test.png', dpi=300, bbox_inches='tight')
plt.show()

# prediction error distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, model_name in enumerate(['Linear Regression', 'Random Forest', 'SVM']):
    errors = y_test.values - predictions[model_name]['test']
    axes[idx].hist(errors, bins=30, edgecolor='black', alpha=0.7)
    axes[idx].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[idx].set_xlabel('Prediction Error', fontweight='bold')
    axes[idx].set_ylabel('Frequency', fontweight='bold')
    axes[idx].set_title(f'{model_name} (MAE = {mean_absolute_error(y_test, predictions[model_name]["test"]):.4f})', 
                       fontsize=12, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_error_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# target distribution
fig, ax = plt.subplots(figsize=(10, 6))
all_targets = pd.concat([y_train, y_val, y_test])
ax.hist(all_targets, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(all_targets.mean(), color='red', linestyle='--', linewidth=2, 
           label=f'Mean = {all_targets.mean():.4f}')
ax.axvline(all_targets.median(), color='green', linestyle='--', linewidth=2, 
           label=f'Median = {all_targets.median():.4f}')
ax.set_xlabel('Diabetes Deaths per 1k Population', fontweight='bold')
ax.set_ylabel('Frequency', fontweight='bold')
ax.set_title('Distribution of Diabetes Risk (Target Variable)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# feature correlation heatmap (top features)
fig, ax = plt.subplots(figsize=(14, 10))
top_20_features = rf_importance.head(20)['Feature'].tolist()
correlation_matrix = X_train[top_20_features].corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Correlation Heatmap - Top 20 Most Important Features', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

def analyze_county(county_name, state_name, model_type='Random Forest'):
    county_data = full_df[(full_df['County'] == county_name) & (full_df['State'] == state_name)]
    if len(county_data) == 0:
        print(f"County '{county_name}, {state_name}' not found in dataset.")
        return None

    county_data = county_data.iloc[0]
    X_county_raw = county_data[feature_cols].values.reshape(1, -1)
    X_county_scaled = scaler.transform(X_county_raw)
    actual_risk = county_data[target_col]

    print(f"ANALYSIS FOR {county_name}, {state_name}")

    # --- Prediction ---
    if model_type == 'Random Forest':
        predicted_risk = rf_model.predict(X_county_raw)[0]
    elif model_type == 'Linear Regression':
        predicted_risk = lr_model.predict(X_county_scaled)[0]
    else: # SVM
        predicted_risk = svm_model.predict(X_county_scaled)[0]

    print()
    print(f"Actual Risk: {actual_risk:.4f} deaths per 1k")
    print(f"Predicted Risk ({model_type}): {predicted_risk:.4f} deaths per 1k")
    print(f"Prediction Error: {abs(actual_risk - predicted_risk):.4f}")

    # feature contributions
    if model_type == 'Random Forest':
        print()
        print("Computing SHAP feature contributions...")
        explainer = shap.TreeExplainer(rf_model)
        shap_values = explainer.shap_values(X_county_raw)

        shap_df = pd.DataFrame({
            'Feature': feature_cols,
            'Feature Value': X_county_raw[0],
            'SHAP Value': shap_values[0]
        }).sort_values('SHAP Value', key=abs, ascending=False)

        print()
        print(f"Top 15 Contributing Factors (SHAP):")
        print(shap_df.head(15)[['Feature', 'Feature Value', 'SHAP Value']].to_string(index=False))
        
        top_contrib = shap_df.head(15)
        fig, ax = plt.subplots(figsize=(12, 8))
        colors = ['red' if x < 0 else 'green' for x in top_contrib['SHAP Value']]
        ax.barh(range(len(top_contrib)), top_contrib['SHAP Value'].values, color=colors, alpha=0.7)
        ax.set_yticks(range(len(top_contrib)))
        ax.set_yticklabels(top_contrib['Feature'].values)
        ax.set_xlabel('SHAP Value (Impact on Prediction)', fontweight='bold')
        ax.set_title(f'Top Risk Factors for {county_name}, {state_name}\n{model_type} Model (SHAP-based)',
                     fontsize=14, fontweight='bold')
        ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'county_analysis_{county_name.replace(" ", "_")}_{state_name}_shap.png',
                    dpi=300, bbox_inches='tight')
        plt.show()

        shap_df.to_csv(f'county_analysis_{county_name.replace(" ", "_")}_{state_name}_shap.csv', index=False)

    elif model_type == 'Linear Regression':
        feature_importance = np.abs(lr_model.coef_)
        contributions = []
        for i, f in enumerate(feature_cols):
            val = X_county_scaled[0, i]
            contributions.append({
                'Feature': f,
                'Value (scaled)': val,
                'Importance': feature_importance[i],
                'Contribution': val * feature_importance[i]
            })
        contrib_df = pd.DataFrame(contributions).sort_values('Contribution', key=abs, ascending=False)

        print()
        print(f"Top 15 Contributing Factors:")
        print(contrib_df.head(15)[['Feature', 'Value (scaled)', 'Contribution']].to_string(index=False))

        top_contrib = contrib_df.head(15)
        fig, ax = plt.subplots(figsize=(12, 8))
        colors = ['red' if x < 0 else 'green' for x in top_contrib['Contribution']]
        ax.barh(range(len(top_contrib)), top_contrib['Contribution'].values, color=colors, alpha=0.7)
        ax.set_yticks(range(len(top_contrib)))
        ax.set_yticklabels(top_contrib['Feature'].values)
        ax.set_xlabel('Contribution to Risk', fontweight='bold')
        ax.set_title(f'Top Risk Factors for {county_name}, {state_name}\n{model_type} Model',
                     fontsize=14, fontweight='bold')
        ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'county_analysis_{county_name.replace(" ", "_")}_{state_name}_lr.png',
                    dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
# example county-level analysis
random_idx = np.random.randint(0, len(test_identifiers))
random_county = test_identifiers.iloc[random_idx]
analyze_county(random_county['County'], random_county['State'], model_type='Random Forest')

In [ ]:
# export data for webapp

full_X = full_df[feature_cols]
full_predictions = rf_model.predict(full_X)

export_df = full_df[['County', 'State']].copy()
export_df['actual_risk'] = full_df['target']
export_df['predicted_risk'] = full_predictions
export_df['prediction_error'] = np.abs(export_df['actual_risk'] - export_df['predicted_risk'])

for col in feature_cols:
    export_df[col] = full_df[col]

export_df['risk_rank'] = export_df['predicted_risk'].rank(ascending=False, method='min').astype(int)
export_df['risk_percentile'] = (export_df['risk_rank'] / len(export_df) * 100).round(1)

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(full_X)

for i, feature in enumerate(feature_cols):
    export_df[f'shap_{feature}'] = shap_values[:, i]

export_df.to_csv('webapp_data.csv', index=False)

feature_metadata = pd.DataFrame({
    'feature': feature_cols,
    'rf_importance': rf_model.feature_importances_
}).sort_values('rf_importance', ascending=False)
feature_metadata.to_csv('feature_metadata.csv', index=False)